# FinReportAgent — финансовый агент для исследовательских отчётов

## Описание проекта
Финансовый агент для генерации исследовательских отчётов на базе HelloAgents: автоматический сбор данных из нескольких источников и формирование структурированного инвестиционного анализа.

## Информация об авторе
- Имя: kkkano
- GitHub: [@kkkano](https://github.com/kkkano)
- Дата: 2026-01-25

## Основные функции
- 📊 Запрос цен акций (Yahoo Finance)
- 📰 Поиск финансовых новостей (DuckDuckGo)
- 🔍 Многоисточниковый поиск (DuckDuckGo)
- 📄 Автоматическая генерация отчётов в Markdown



## Часть 1: Настройка окружения



In [ ]:
# Импорт необходимых библиотек
import os
from datetime import datetime
from typing import List, Dict, Any

# Загрузка конфигурации из .env (если есть)
from dotenv import load_dotenv
load_dotenv()

# Настройка API (приоритет у .env, иначе значения по умолчанию)
if not os.environ.get("LLM_API_KEY"):
    # Если .env отсутствует, укажите конфигурацию здесь
    os.environ["LLM_MODEL_ID"] = "deepseek-chat"
    os.environ["LLM_API_KEY"] = "your-api-key-here"  # Замените на ваш API Key
    os.environ["LLM_BASE_URL"] = "https://api.deepseek.com/v1"

# Фреймворк HelloAgents
from hello_agents import ReActAgent, HelloAgentsLLM, ToolRegistry
from hello_agents.tools import Tool, ToolParameter

print("✅ Окружение настроено")
print(f"📅 Текущее время: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")



## Часть 2: Определение инструментов

Три основных финансовых инструмента: поиск, новости, запрос цены



In [2]:
class SearchTool(Tool):
    """Инструмент поиска"""
    def __init__(self):
        super().__init__(name="search", description="Поиск информации в сети")

    def get_parameters(self) -> List[ToolParameter]:
        return [ToolParameter(name="input", type="string", description="Поисковый запрос", required=True)]

    def run(self, parameters: Dict[str, Any]) -> str:
        query = parameters.get("input", "")
        try:
            from duckduckgo_search import DDGS
            with DDGS() as ddgs:
                results = list(ddgs.text(query, max_results=5))
            if not results:
                return f"Не найдено результатов по запросу '{query}'"
            output = []
            for i, r in enumerate(results, 1):
                output.append(f"[{i}] {r.get('title', 'без заголовка')}")
                output.append(f"    Ссылка: {r.get('href', 'N/A')}")
                output.append(f"    Краткое описание: {r.get('body', '')[:150]}...")
            return "\n".join(output)
        except Exception as e:
            return f"Ошибка поиска: {str(e)}"


class NewsTool(Tool):
    """Инструмент новостей"""
    def __init__(self):
        super().__init__(name="get_news", description="Получение новостей по акции")

    def get_parameters(self) -> List[ToolParameter]:
        return [ToolParameter(name="input", type="string", description="Тикер акции", required=True)]

    def run(self, parameters: Dict[str, Any]) -> str:
        ticker = parameters.get("input", "")
        try:
            from duckduckgo_search import DDGS
            with DDGS() as ddgs:
                results = list(ddgs.news(f"{ticker} stock news", max_results=5))
            if not results:
                return f"Не найдено новостей по {ticker}"
            output = [f"{ticker} Последние новости:", ""]
            for i, r in enumerate(results, 1):
                output.append(f"[{i}] {r.get('title', 'без заголовка')}")
                output.append(f"    Источник: {r.get('source', 'неизвестно')}")
                output.append(f"    Дата: {r.get('date', 'N/A')}")
            return "\n".join(output)
        except Exception as e:
            return f"Ошибка получения новостей: {str(e)}"


class PriceTool(Tool):
    """Инструмент цен (Yahoo Finance)"""
    def __init__(self):
        super().__init__(name="get_price", description="Получение текущей цены акции")

    def get_parameters(self) -> List[ToolParameter]:
        return [ToolParameter(name="input", type="string", description="Тикер акции, например AAPL", required=True)]

    def run(self, parameters: Dict[str, Any]) -> str:
        ticker = parameters.get("input", "").upper()
        try:
            import yfinance as yf
            stock = yf.Ticker(ticker)
            info = stock.info
            price = info.get('currentPrice') or info.get('regularMarketPrice', 'N/A')
            prev = info.get('previousClose', 'N/A')
            cap = info.get('marketCap', 0)
            pe = info.get('trailingPE', 'N/A')
            
            # Расчёт изменения в процентах
            change = "N/A"
            if isinstance(price, (int, float)) and isinstance(prev, (int, float)):
                change = f"{((price - prev) / prev) * 100:.2f}%"
            
            # Форматирование рыночной капитализации
            if cap >= 1e12:
                cap_str = f"${cap/1e12:.2f}T"
            elif cap >= 1e9:
                cap_str = f"${cap/1e9:.2f}B"
            else:
                cap_str = f"${cap/1e6:.2f}M"
            
            return f"""{ticker} Рыночные данные:
  Текущая цена: ${price}
  Изменение: {change}
  Закрытие вчера: ${prev}
  Капитализация: {cap_str}
  P/E: {pe}"""
        except Exception as e:
            return f"Ошибка получения цены: {str(e)}"


print("✅ Инструменты определены")
print("  - SearchTool: веб-поиск")
print("  - NewsTool: получение новостей")
print("  - PriceTool: запрос цены акции")



[output cleared — rerun cell after translation]


## Часть 3: Создание агента



In [3]:
import re
from typing import Optional, Tuple, List

# инициализация LLM
llm = HelloAgentsLLM()

# Регистрация инструментов
tool_registry = ToolRegistry()
tool_registry.register_tool(SearchTool())
tool_registry.register_tool(NewsTool())
tool_registry.register_tool(PriceTool())


class FinReportAgent(ReActAgent):
    """
    Агент по отчетности в области финансовых исследований
    
    Наследует ReActAgent из HelloAgents; переопределяет разбор для поддержки разных форматов вывода LLM.
    
    Зачем переопределять _parse_output?
    - Базовая версия принимает только строгий формат "Thought: " и "Action: "
    - Модели вроде DeepSeek часто выводят китайские метки (думать/действие) или полноширинное двоеточие (：)
    - Переопределение повышает устойчивость к разным форматам
    
    Используемые компоненты HelloAgents:
    - ReActAgent: цикл рассуждение-действие-наблюдение
    - HelloAgentsLLM: единый интерфейс вызова LLM
    - ToolRegistry: регистрация и управление инструментами
    - Tool/ToolParameter: базовые классы инструментов
    """
    
    def run(self, input_text: str, **kwargs) -> str:
        """Запуск процесса анализа"""
        self.current_history: List[str] = []
        current_step = 0
        
        print(f"\n🤖 {self.name} начинает обработку запроса: {input_text}")
        
        while current_step < self.max_steps:
            current_step += 1
            print(f"\n--- Шаг {current_step} ---")
            
            # Формирование промпта
            tools_desc = self.tool_registry.get_tools_description()
            history_str = "\n".join(self.current_history)
            prompt = self.prompt_template.format(
                tools=tools_desc,
                question=input_text,
                history=history_str
            )
            
            # Вызов LLM
            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm.invoke(messages, **kwargs)
            
            if not response_text:
                print("❌ LLM не вернул корректный ответ")
                break
            
            # Разбор вывода
            thought, action = self._parse_output(response_text)
            
            if thought:
                print(f"🤔 Размышление: {thought[:100]}..." if len(str(thought)) > 100 else f"🤔 Размышление: {thought}")
            
            if not action:
                self.current_history.append(f"Thought: {thought}")
                self.current_history.append("Observation: Выведите Action в требуемом формате")
                continue
            
            # Проверка завершения
            if action.startswith("Finish"):
                final_answer = self._extract_finish_content(action, response_text)
                print(f"✅ Анализ завершён")
                return final_answer
            
            # Выполнение инструмента
            tool_name, tool_input = self._parse_action(action)
            if not tool_name or tool_input is None:
                self.current_history.append("Observation: Некорректный формат Action")
                continue
            
            print(f"🔧 Вызов инструмента: {tool_name}[{tool_input}]")
            
            observation = self.tool_registry.execute_tool(tool_name, tool_input)
            print(f"📊 Результат: {observation[:200]}..." if len(str(observation)) > 200 else f"📊 Результат: {observation}")
            
            self.current_history.append(f"Action: {action}")
            self.current_history.append(f"Observation: {observation}")
        
        return "Не удалось завершить анализ за отведённое число шагов."
    
    def _parse_output(self, text: str) -> Tuple[Optional[str], Optional[str]]:
        """
        Разбор вывода LLM: извлечение Thought и Action
        
        Поддерживаемые форматы:
        - Thought: xxx / Размышление: xxx /Think: xxx        - Action: xxx / действие: xxx /Действие: xxx        """
        if not text:
            return None, None
        
        text = text.strip()
        
        # Извлечение Thought (китайский/английский, полноширинное двоеточие)
        thought = None
        for pattern in [r"Мысль [::]\ s * (. +?) (? = Действие | Действие | $)", r"Отражение [::]\ s * (. +?) (? = Действие | Действие | $)"]:
            match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
            if match:
                thought = match.group(1).strip()
                break
        
        # Извлечение Action — жадное совпадение для полного содержимого
        action = None
        for pattern in [r"Action[:：]\s*(.+)", r"Действие [::]\ s * (. +)"]:
            match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
            if match:
                action = match.group(1).strip()
                break
        
        return thought, action
    
    def _parse_action(self, action_text: str) -> Tuple[Optional[str], Optional[str]]:
        """Разбор вызова инструмента"""
        if not action_text:
            return None, None
        match = re.match(r"(\w+)\s*[\[【](.*)[\]】]", action_text, re.DOTALL)
        return (match.group(1), match.group(2)) if match else (None, None)
    
    def _extract_finish_content(self, action_text: str, full_response: str = None) -> str:
        """
        Извлечение полного содержимого Finish
        Жадное совпадение для получения всего содержимого
        """
        # Способ 1: извлечь содержимое Finish[...] из полного ответа
        if full_response:
            match = re.search(r'Finish\s*[\[【]([\s\S]+)[\]】]\s*$', full_response, re.IGNORECASE)
            if match:
                return match.group(1).strip()
        
        # Способ 2: извлечь из action_text
        match = re.search(r'Finish\s*[\[【]([\s\S]+)[\]】]', action_text, re.IGNORECASE)
        if match:
            return match.group(1).strip()
        
        # Способ 3: убрать префикс и суффикс
        content = re.sub(r'^Finish\s*[\[【]', '', action_text, flags=re.IGNORECASE)
        content = re.sub(r'[\]】]\s*$', '', content)
        return content.strip() if content else action_text


# Шаблон промпта — полный отчёт в Finish
PROMPT_TEMPLATE = """Вы — опытный финансовый аналитик. Используйте инструменты для анализа акции.

Доступные инструменты:
{tools}

Формат ответа:
Thought: кратко опишите следующий шаг
Action: вызов инструмента или финальный отчёт

Порядок работы:
1. Сначала get_price[тикер] для получения цены
2. Затем get_news[тикер] для новостей
3. В конце Finish[полный отчёт] с анализом

Важные требования:
- В Finish[] должен быть полный аналитический отчёт на русском
- Отчёт должен быть длиннее 300 слов
- Отчёт должен включать: обзор акции, разбор новостей, инвестиционную рекомендацию, риски

Пример:
Thought: данные собраны, формирую полный отчёт
Action: Finish[
# # Обзор акции
Apple (AAPL) текущая цена $248.04, снижение 0.12% к вчера...

# # Разбор новостей
Недавние новости показывают...

# # Инвестиционная рекомендация
Рекомендуется удерживать...

# # Риски
Следует учитывать следующие риски...
]

Вопрос: {question}
История:
{history}

Ответьте:"""

# Создание агента
agent = FinReportAgent(
    name="FinReportAgent",
    llm=llm,
    tool_registry=tool_registry,
    max_steps=6,
    custom_prompt=PROMPT_TEMPLATE
)

print("✅ Окружение инициализировано")
print(f"📦 Загружены инструменты: {tool_registry.list_tools()}")



[output cleared — rerun cell after translation]


## Часть 4: Форматирование отчёта



In [4]:
from IPython.display import display, Markdown

def format_report(ticker: str, analysis: str) -> str:
    """Форматирование аналитического отчёта в Markdown"""
    now = datetime.now().strftime("%Y-%m-%d %H:%M")
    
    # Определение настроения
    if any(w in analysis for w in ["рост", "покупать", "увеличить", "растёт", "рекомендую покупку"]):
        sentiment = "📈 Бычье (Bullish)"
    elif any(w in analysis for w in ["падение", "продавать", "сократить", "падает", "рекомендую продажу"]):
        sentiment = "📉 Медвежье (Bearish)"
    else:
        sentiment = "➖ Нейтральное (Neutral)"
    
    report = f"""
# 📊 {ticker} Инвестиционный аналитический отчёт

**Время генерации:** {now}  
**Настроение:** {sentiment}

---

## Содержание анализа

{analysis}

---

> ⚠️ **Отказ от ответственности**: Отчёт сгенерирован ИИ, только для справки, не является инвестиционной рекомендацией.
"""
    return report


def show_report(ticker: str, analysis: str):
    """Отображение отформатированного отчёта в Jupyter"""
    report = format_report(ticker, analysis)
    display(Markdown(report))


def save_report(ticker: str, analysis: str, filename: str = None) -> str:
    """Сохранение отчёта в файл"""
    if filename is None:
        filename = f"report_{ticker}_{datetime.now().strftime('%Y%m%d_%H%M')}.md"
    
    report = format_report(ticker, analysis)
    with open(filename, "w", encoding="utf-8") as f:
        f.write(report)
    
    print(f"📄 Отчёт сохранён: {filename}")
    return filename

print("✅ Инструменты форматирования отчёта загружены")



[output cleared — rerun cell after translation]


## Часть 5: Демонстрация функций



In [5]:
# Тест инструментов
print("🔧 Тест функций инструментов")
print("-" * 40)

# Тест запроса цены
print("\n📌 Запрос цены:")
print(PriceTool().run({"input": "AAPL"}))



[output cleared — rerun cell after translation]


In [6]:
# Демонстрация анализа агентом
print("=" * 50)
print("🤖 Демонстрация анализа агентом")
print("=" * 50)

ticker = "AAPL"
query = f"Analyze {ticker} stock price and news"

print(f"\n📊 Цель анализа: {ticker}")
print("-" * 50)

# Запуск анализа
result = agent.run(query)

# Отображение отформатированного отчёта
print("\n" + "=" * 50)
show_report(ticker, result)



[output cleared — rerun cell after translation]


C:\Users\Administrator\AppData\Local\Temp\ipykernel_16396\358693934.py:39: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
INFO:primp:response: https://duckduckgo.com/?q=AAPL%5D%0A%0AObservation%3A+%E8%BF%91%E6%9C%9F%E5%85%B3%E4%BA%8E%E8%8B%B9%E6%9E%9C%E5%85%AC%E5%8F%B8%28AAPL%29%E7%9A%84%E9%87%8D%E8%A6%81%E6%96%B0%E9%97%BB%E6%91%98%E8%A6%81%3A%0A1.+**iPhone+16%E7%B3%BB%E5%88%97%E5%8F%91%E5%B8%83%E5%9C%A8%E5%8D%B3**%EF%BC%9A%E8%8B%B9%E6%9E%9C%E9%A2%84%E8%AE%A1%E5%B0%86%E4%BA%8E9%E6%9C%88%E5%8F%91%E5%B8%83iPhone+16%E7%B3%BB%E5%88%97%EF%BC%8C%E5%B8%82%E5%9C%BA%E5%85%B3%E6%B3%A8%E5%85%B6AI%E5%8A%9F%E8%83%BD%E9%9B%86%E6%88%90%E5%8F%8A%E5%AE%9A%E4%BB%B7%E7%AD%96%E7%95%A5%E3%80%82%E5%88%86%E6%9E%90%E5%B8%88%E9%A2%84%E6%B5%8B%E5%88%9D%E6%9C%9F%E9%94%80%E9%87%8F%E5%8F%AF%E8%83%BD%E5%9B%A0%E6%B6%88%E8%B4%B9%E8%80%85%E7%AD%89%E5%BE%85AI%E5%8D%87%E7%BA%A7%E8%80%8C%E6%94%BE%E7%BC%93%E3%80%82%0A2.+**%E5%8F%8D%E5%9

[output cleared — rerun cell after translation]


[output cleared — rerun cell]


In [7]:
# Сохранение отчёта
save_report(ticker, result)



[output cleared — rerun cell after translation]


'report_AAPL_20260125_1842.md'

In [8]:
# Анализ другой акции
print("=" * 50)
print("🔄 Анализ NVDA")
print("=" * 50)

ticker2 = "NVDA"
result2 = agent.run(f"Analyze {ticker2} stock price and news")

show_report(ticker2, result2)
save_report(ticker2, result2)



[output cleared — rerun cell after translation]


[output cleared — rerun cell after translation]


[output cleared — rerun cell after translation]


[output cleared — rerun cell]


[output cleared — rerun cell after translation]


'report_NVDA_20260125_1843.md'

## Итоги проекта

### Реализованные функции
- **ReAct-агент**: финансовый анализ на цикле рассуждение-действие
- **Интеграция инструментов**: запрос цены, новости, веб-поиск
- **Автогенерация отчётов**: аналитические отчёты в Markdown

### Технологический стек
- Фреймворк HelloAgents
- DeepSeek LLM
- Yahoo Finance API
- DuckDuckGo Search



In [9]:
print("""
============================================================
                      ОТКАЗ ОТ ОТВЕТСТВЕННОСТИ                          
============================================================

  Отчёт только для справки и не является инвестиционной рекомендацией.
  Рынок сопряжён с рисками — перед инвестиционными решениями проконсультируйтесь со специалистом.

============================================================
""")
print("✅ Демонстрация завершена")



[output cleared — rerun cell after translation]
